# 自由现金流（FCF）模型实战教程

**目标**：用模拟财务数据，走通"自由现金流计算 → DCF估值"的完整流程。

**特色**：完全自包含，使用模拟数据，无需外部 API 依赖，立即可运行。

**依赖**：仅需要 `pandas numpy matplotlib`

---

## 不同行业使用 FCF 的注意事项

### 银行业
**问题**：银行业现金流表结构与普通企业不同
- 资本支出概念不适用（银行的核心资产是贷款和投资）
- FCF 可能波动剧烈，不适合用传统公式分析

**替代方案**：
- 使用 **股东自由现金流 (FCFE)** 或 **公司自由现金流 (FCFF)**
- 关注 **净利息收入** 和 **风险调整后资本回报率 (RAROC)**
- 估值推荐：使用 **剩余收益模型 (RIM)** 或 **股利折现模型 (DDM)**

### 高增长行业（如科技、生物医药）
**问题**：
- 高研发投入导致 FCF 为负，但未来前景看好
- 资本支出主要用于增长而非维持运营

**调整方案**：
- 将 **研发支出资本化**（而非费用化），再计算 FCF
- 使用 **自由现金流收益率 (FCF Yield)**，关注 FCF 相对于市值的比率
- 估值推荐：使用 **实物期权定价模型**，捕捉增长期权的价值

### 周期性行业（如原材料、航运）
**问题**：
- FCF 随周期剧烈波动，单期数据不可靠
- 终值占比过高，预测困难

**调整方案**：
- 使用 **多周期平均 FCF** 而非单期
- 应用 **蒙特卡洛模拟** 对未来情景建模
- 估值推荐：使用 **实物期权** 或 **相对估值法（如 EV/EBITDA）**

### 零售/消费品行业
**特点**：
- FCF 稳定，最适合传统 DCF 分析
- 关注同店增长率和库存周转

**调整方案**：
- 分析 **自由现金流/营收比** 的时间趋势
- 考虑 **营运资本变动** 对 FCF 的影响
- 估值推荐：标准 DCF 模型 + 敏感性分析

## 核心概念

> 自由现金流 = 企业经营产生的现金 - 维持经营必需的现金支出

**基础公式**：
```
FCF = 经营活动现金流 - 资本支出
```

**更精确的公式**：
```
FCF = 经营活动现金流 - 资本支出 + 利息收入 - 利息支出
```

**行业调整公式**：
- **高增长科技**：`调整后FCF = 经营现金流 - 维持性资本支出 + 研发支出资本化`
- **银行**：使用 RAROC 或 DDM，不适用传统 FCF
- **周期行业**：`平滑FCF = 过去5-10年FCF的加权平均`

## 创建模拟财务数据

为演示目的，我们创建一家稳定增长消费品企业的模拟财务数据：
- 5 年历史数据（2020-2024）
- 3 年预测数据（2025-2027）
- 包含三大报表：现金流量表、利润表、资产负债表

In [ ]:
# 导入依赖（仅需 pandas + numpy + matplotlib）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# 中文字体配置：Windows 优先微软雅黑，缺字时回退
mpl.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False  # 负号显示正常

# notebook 内联绘图
%matplotlib inline

print(f'pandas {pd.__version__} | numpy {np.__version__} | matplotlib {mpl.__version__}')
print('中文字体:', mpl.rcParams['font.sans-serif'][0])

In [ ]:
def create_mock_company_financials(company_name: str = '模拟消费品公司') -> dict:
    """创建完整的模拟财务数据
    
    Args:
        company_name: 公司名称
    
    Returns:
        包含三大报表的字典
    """
    years = list(range(2020, 2028))  # 2020-2027 (8年)
    
    # 基础参数（2020年）
    base_revenue = 160.0  # 亿元（调高使估值更合理）
    net_profit_margin = 0.18
    revenue_growth_rates = [0.08, 0.10, 0.09, 0.07, 0.08,  # 历史增长率
                           0.08, 0.07, 0.06]  # 预测增长率
    
    data = []
    current_revenue = base_revenue
    
    for i, year in enumerate(years):
        # 营业收入（历史有波动，预测平滑）
        if i < 5:
            volatility = np.random.uniform(-0.02, 0.02)
            current_revenue *= (1 + revenue_growth_rates[i] + volatility)
        else:
            current_revenue *= (1 + revenue_growth_rates[i])
        
        # 计算其他财务指标
        net_profit = current_revenue * net_profit_margin
        
        # 现金流量表数据
        operating_cf = net_profit * 0.95  # 经营现金流略高于净利润
        capex = current_revenue * 0.08  # 资本支出占营收的8%
        fcf = operating_cf - capex
        
        # 资产负债表数据
        total_assets = current_revenue * 1.5
        net_assets = total_assets * 0.6
        
        # 数据类型标记
        data_type = '历史' if i < 5 else '预测'
        
        data.append({
            '年份': year,
            '数据类型': data_type,
            # 利润表
            '营业收入(亿)': round(current_revenue, 2),
            '净利润(亿)': round(net_profit, 2),
            # 现金流量表
            '经营活动现金流(亿)': round(operating_cf, 2),
            '资本支出(亿)': round(capex, 2),
            '自由现金流(亿)': round(fcf, 2),
            # 资产负债表
            '总资产(亿)': round(total_assets, 2),
            '净资产(亿)': round(net_assets, 2),
        })
    
    financials = pd.DataFrame(data)
    
    # 分离为三个报表
    cash_flow = financials[['年份', '数据类型', '经营活动现金流(亿)', '资本支出(亿)', '自由现金流(亿)']]
    profit = financials[['年份', '数据类型', '营业收入(亿)', '净利润(亿)']]
    balance = financials[['年份', '数据类型', '总资产(亿)', '净资产(亿)']]
    
    return {
        'name': company_name,
        'cash_flow': cash_flow,
        'profit': profit, 
        'balance': balance,
        'full_data': financials
    }

# 创建模拟公司数据
mock_company = create_mock_company_financials('优质消费品公司')
print(f"模拟公司：{mock_company['name']}")
print(f"数据期间：{mock_company['cash_flow']['年份'].min()} - {mock_company['cash_flow']['年份'].max()}")
print(f"总期数：{len(mock_company['full_data'])} 期")

mock_company['full_data']

## 计算自由现金流

**基础公式**：`FCF = 经营活动现金流 - 资本支出`

In [ ]:
def calculate_fcf(cash_flow_df: pd.DataFrame) -> pd.DataFrame:
    """计算自由现金流
    
    Args:
        cash_flow_df: 现金流量表 DataFrame
    
    Returns:
        包含 FCF 的 DataFrame
    """
    operating_cf = cash_flow_df['经营活动现金流(亿)']
    capex = cash_flow_df['资本支出(亿)']
    fcf = operating_cf - capex
    
    result_df = cash_flow_df.copy()
    result_df['自由现金流(亿)'] = fcf
    return result_df

fcf_df = calculate_fcf(mock_company['cash_flow'])
print("FCF 计算成功！")
print(f"最近一期 FCF: {fcf_df['自由现金流(亿)'].iloc[-1]:,.2f} 亿元")
fcf_df.tail(3)

## 可视化 FCF 趋势

In [ ]:
def plot_fcf_trend(fcf_df: pd.DataFrame, company_name: str):
    """绘制 FCF 趋势图
    
    Args:
        fcf_df: FCF DataFrame（包含'自由现金流(亿)'列）
        company_name: 公司名称
    """
    if fcf_df.empty:
        print("无数据可绘制")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. FCF 绝对值趋势
    axes[0].plot(range(len(fcf_df)), fcf_df['自由现金流(亿)'], 
                    marker='o', linewidth=2, color='steelblue')
    axes[0].set_title(f'{company_name} - 自由现金流趋势')
    axes[0].set_ylabel('金额（亿元）')
    axes[0].set_xlabel('期数')
    axes[0].grid(True, alpha=0.3)
    
    # 2. FCF 增长率
    growth_rates = fcf_df['自由现金流(亿)'].pct_change()
    colors = ['green' if x > 0 else 'red' for x in growth_rates]
    axes[1].bar(range(len(fcf_df)), growth_rates, color=colors, alpha=0.7)
    axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
    axes[1].set_title('FCF 增长率')
    axes[1].set_ylabel('增长率')
    axes[1].set_xlabel('期数')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_fcf_trend(fcf_df, mock_company['name'])

## DCF 估值模型

### 估值原理

> 企业价值 = 未来自由现金流的现值总和

**核心公式**：
```
企业价值 = ∑(未来FCF / (1+折现率)^t) + 终值
股权价值 = 企业价值 - 净债务
每股价值 = 股权价值 / 股本总数
```

**三阶段 DCF 模型**：
1. 预测期价值 (通常3-5年)：未来FCF的现值总和
2. 终值 (永续增长期)：预测期后按固定增长率永续增长
3. 企业价值 = 预测期价值 + 终值

In [ ]:
def calculate_avg_growth_rate(fcf_series: pd.Series) -> float:
    """计算FCF历史平均增长率
    
    Args:
        fcf_series: FCF时间序列
    
    Returns:
        平均增长率
    """
    growth_rates = []
    for i in range(1, len(fcf_series)):
        if fcf_series.iloc[i-1] > 0:
            growth_rates.append((fcf_series.iloc[i] - fcf_series.iloc[i-1]) / fcf_series.iloc[i-1])
    return np.mean(growth_rates) if growth_rates else 0.02

def dcf_valuation(current_fcf: float,
                avg_growth_rate: float,
                forecast_years: int = 3,
                discount_rate: float = 0.10,
                terminal_growth_rate: float = 0.03,
                net_debt: float = 50.0,
                total_shares: float = 10.0) -> dict:
    """DCF 估值模型
    
    Args:
        current_fcf: 当前年度FCF (亿元)
        avg_growth_rate: 历史平均增长率
        forecast_years: 预测年数
        discount_rate: 折现率 (WACC)
        terminal_growth_rate: 终期增长率 (通常2-3%)
        net_debt: 净债务 (亿元)
        total_shares: 总股本 (亿股)
    
    Returns:
        估值结果字典
    """
    # 保守起见，取历史增长率的一半
    base_growth_rate = max(avg_growth_rate * 0.5, terminal_growth_rate)
    
    # 预测期现金流（增长率逐年递减）
    forecasted_fcf = []
    growth_schedule = [base_growth_rate, base_growth_rate * 0.8, base_growth_rate * 0.6]
    
    forecast_fcf = current_fcf
    for i in range(forecast_years):
        growth_rate = growth_schedule[i] if i < len(growth_schedule) else terminal_growth_rate
        forecast_fcf *= (1 + growth_rate)
        forecasted_fcf.append(forecast_fcf)
    
    # 计算预测期现值
    pv_forecast = sum(fcf / ((1 + discount_rate) ** (i + 1)) for i, fcf in enumerate(forecasted_fcf))
    
    # 计算终值
    terminal_fcf = forecasted_fcf[-1] * (1 + terminal_growth_rate)
    terminal_value = terminal_fcf / (discount_rate - terminal_growth_rate)
    pv_terminal = terminal_value / ((1 + discount_rate) ** forecast_years)
    
    # 企业价值
    enterprise_value = pv_forecast + pv_terminal
    
    # 股权价值
    equity_value = enterprise_value - net_debt
    
    # 每股价值
    share_value = equity_value / total_shares
    
    return {
        '预测期现金流(亿)': forecasted_fcf,
        '预测期现值(亿)': [fcf / ((1 + discount_rate) ** (i + 1)) for i, fcf in enumerate(forecasted_fcf)],
        '预测期现值总计(亿)': round(pv_forecast, 2),
        '终期现金流(亿)': round(terminal_fcf, 2),
        '终值(亿)': round(terminal_value, 2),
        '终值现值(亿)': round(pv_terminal, 2),
        '企业价值(亿)': round(enterprise_value, 2),
        '净债务(亿)': net_debt,
        '股权价值(亿)': round(equity_value, 2),
        '总股本(亿股)': total_shares,
        '每股价值(元)': round(share_value, 2),
        '折现率': discount_rate,
        '终期增长率': terminal_growth_rate,
        '历史平均增长率': round(avg_growth_rate, 3)
    }

## 多公司对比分析

In [ ]:
def create_multiple_companies(count: int = 5) -> list:
    """创建多个模拟公司
    
    Args:
        count: 公司数量
    
    Returns:
        公司列表
    """
    companies = []
    company_types = [
        '优质消费品', '稳定制造业', '成长科技', '周期性企业', '传统金融'
    ]
    
    for i in range(count):
        name = f"{company_types[i % len(company_types)]}公司{i+1}"
        np.random.seed(i * 42)  # 确保可重复性
        companies.append(create_mock_company_financials(name))
    
    return companies

companies = create_multiple_companies(5)

# 对所有公司进行 DCF 估值
valuation_results = []
for company in companies:
    historical_data = company['cash_flow'][company['cash_flow']['数据类型'] == '历史'].tail(3)
    current_fcf = historical_data['自由现金流(亿)'].iloc[-1]
    avg_growth = calculate_avg_growth_rate(historical_data['自由现金流(亿)'])
    
    result = dcf_valuation(
        current_fcf,
        avg_growth,
        forecast_years=3,
        discount_rate=0.10,
        terminal_growth_rate=0.03,
        net_debt=20.0,
        total_shares=8.0
    )
    result['公司名称'] = company['name']
    valuation_results.append(result)

valuation_df = pd.DataFrame(valuation_results)
print("多公司 DCF 估值结果：")
valuation_df[['公司名称', '每股价值(元)', '企业价值(亿)', '股权价值(亿)', '历史平均增长率']].round(2)

## DCF 估值可视化分析

In [ ]:
def plot_dcf_breakdown(valuation_result: dict, current_price: float = 28.0):
    """绘制DCF估值分解图
    
    Args:
        valuation_result: DCF估值结果
        current_price: 当前股价
    """
    fig = plt.figure(figsize=(15, 8))
    
    # 1. 现金流分解图
    ax1 = plt.subplot(2, 3, 1)
    years = [f'第{i+1}年' for i in range(len(valuation_result['预测期现金流(亿)']))]
    colors = plt.cm.Blues(range(len(years)))
    
    bars = ax1.bar(years, valuation_result['预测期现金流(亿)'], color=colors, alpha=0.7)
    ax1.set_ylabel('FCF (亿元)')
    ax1.set_title('预测期现金流')
    ax1.grid(True, alpha=0.3)
    
    for bar, val in zip(bars, valuation_result['预测期现金流(亿)']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
                f'{val:.1f}', ha='center', va='bottom', fontsize=9)
    
    # 2. 企业价值分解饼图
    ax2 = plt.subplot(2, 3, 2)
    value_components = [
        valuation_result['预测期现值总计(亿)'],
        valuation_result['终值现值(亿)']
    ]
    labels = [f'预测期现值\n{value_components[0]:.1f}亿', 
              f'终值现值\n{value_components[1]:.1f}亿']
    colors = ['lightblue', 'lightcoral']
    
    ax2.pie(value_components, labels=labels, colors=colors, 
           autopct='%1.1f%%', startangle=90)
    ax2.set_title('企业价值构成')
    
    # 3. 敏感性分析
    ax3 = plt.subplot(2, 3, 3)
    discount_rates = np.linspace(0.08, 0.12, 9)
    share_values = []
    
    for dr in discount_rates:
        result = dcf_valuation(valuation_result['预测期现金流(亿)'][0] * 0.92,
                              valuation_result['历史平均增长率'],
                              discount_rate=dr, terminal_growth_rate=0.03,
                              net_debt=20.0, total_shares=8.0)
        share_values.append(result['每股价值(元)'])
    
    ax3.plot(discount_rates * 100, share_values, 'b-', linewidth=2, marker='o')
    ax3.axhline(y=current_price, color='r', linestyle='--', label=f'当前价 {current_price:.1f}元')
    ax3.set_xlabel('折现率 (%)')
    ax3.set_ylabel('每股价值 (元)')
    ax3.set_title('折现率敏感性分析')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. 估值总结表
    ax4 = plt.subplot(2, 3, 4)
    ax4.axis('off')
    
    safety_margin = (valuation_result['每股价值(元)'] - current_price) / current_price * 100
    
    summary_data = [
        ['项目', '数值'],
        ['企业价值', f"{valuation_result['企业价值(亿)']:.1f} 亿元"],
        ['股权价值', f"{valuation_result['股权价值(亿)']:.1f} 亿元"],
        ['每股价值', f"{valuation_result['每股价值(元)']:.2f} 元"],
        ['当前股价', f"{current_price:.2f} 元"],
        ['安全边际', f"{safety_margin:.1f}%"]
    ]
    
    table = ax4.table(cellText=summary_data[1:], colLabels=summary_data[0],
                     cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    
    for i in range(len(summary_data)):
        for j in range(2):
            cell = table[(i, j)]
            if i == 0:
                cell.set_facecolor('#40466e')
                cell.set_text_props(color='white', weight='bold')
            else:
                cell.set_facecolor('#f1f1f2')
    
    ax4.set_title('估值总结', pad=20)
    
    # 5. 估值结论
    ax5 = plt.subplot(2, 3, 5)
    ax5.axis('off')
    
    intrinsic_value = valuation_result['每股价值(元)']
    safety_margin_val = (intrinsic_value - current_price) / current_price
    
    if safety_margin_val > 0.3:
        conclusion = "绿色 价值低估"
        recommendation = "强烈推荐买入"
    elif safety_margin_val > 0.1:
        conclusion = "黄色 轻度低估"
        recommendation = "可以考虑买入"
    elif safety_margin_val > -0.1:
        conclusion = "白色 合理估值"
        recommendation = "观望为主"
    else:
        conclusion = "红色 价值高估"
        recommendation = "建议规避"
    
    conclusion_text = f"""\n
    **估值结论**\n
    \n
    {conclusion}\n
    \n
    • 内在价值: {intrinsic_value:.2f} 元\n
    • 安全边际: {safety_margin_val*100:.1f}%\n
    • 投资建议: {recommendation}\n
    \n
    **关键假设**\n
    • 折现率: {valuation_result['折现率']*100:.1f}%\n
    • 终期增长率: {valuation_result['终期增长率']*100:.1f}%\n
    """
    
    ax5.text(0.1, 0.5, conclusion_text, fontsize=10, verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 6. 多公司对比
    ax6 = plt.subplot(2, 3, 6)
    ax6.barh(valuation_df['公司名称'], valuation_df['每股价值(元)'], alpha=0.7)
    ax6.axvline(x=current_price, color='r', linestyle='--', label=f'当前价 {current_price:.1f}元')
    ax6.set_xlabel('每股价值 (元)')
    ax6.set_title('多公司估值对比')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 绘制第一个公司的 DCF 分析图
if valuation_results:
    plot_dcf_breakdown(valuation_results[0], current_price=28.0)

## 敏感性分析矩阵

In [ ]:
def sensitivity_analysis_matrix(current_fcf: float, avg_growth: float,
                                discount_rates: list, terminal_rates: list) -> pd.DataFrame:
    """敏感性分析矩阵
    
    Args:
        current_fcf: 当前FCF
        avg_growth: 平均增长率
        discount_rates: 折现率列表
        terminal_rates: 终期增长率列表
    
    Returns:
        敏感性矩阵
    """
    matrix = []
    
    for dr in discount_rates:
        row = {'折现率': f'{dr*100:.1f}%'}
        for tr in terminal_rates:
            result = dcf_valuation(current_fcf, avg_growth,
                                  discount_rate=dr, terminal_growth_rate=tr,
                                  net_debt=20.0, total_shares=8.0)
            row[f'{tr*100:.1f}%'] = result['每股价值(元)']
        matrix.append(row)
    
    df = pd.DataFrame(matrix)
    df.columns = ['折现率'] + [f'{tr*100:.1f}% 终期增长率' for tr in terminal_rates]
    return df

discount_rates = [0.08, 0.09, 0.10, 0.11, 0.12]
terminal_rates = [0.02, 0.025, 0.03, 0.035, 0.04]

# 使用第一个公司的数据进行分析
if valuation_results:
    result = valuation_results[0]
    current_fcf = result['预测期现金流(亿)'][0] * 0.92  # 反推
    avg_growth = result['历史平均增长率']
    
    sensitivity_df = sensitivity_analysis_matrix(current_fcf, avg_growth, discount_rates, terminal_rates)
    print("DCF 估值敏感性分析矩阵 (每股价值，单位：元):")
    sensitivity_df

In [ ]:
def plot_sensitivity_heatmap(sensitivity_df: pd.DataFrame, current_price: float = 28.0):
    """绘制敏感性分析热力图
    
    Args:
        sensitivity_df: 敏感性矩阵
        current_price: 当前股价
    """
    values = sensitivity_df.iloc[:, 1:].values
    safety_margins = (values - current_price) / current_price * 100
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. 每股价值热力图
    im1 = ax1.imshow(values, cmap='RdYlGn', aspect='auto', 
                     vmin=values.min(), vmax=values.max())
    
    ax1.set_xticks(range(len(terminal_rates)))
    ax1.set_yticks(range(len(discount_rates)))
    ax1.set_xticklabels([f'{tr*100:.1f}%' for tr in terminal_rates])
    ax1.set_yticklabels([f'{dr*100:.1f}%' for dr in discount_rates])
    ax1.set_xlabel('终期增长率')
    ax1.set_ylabel('折现率')
    ax1.set_title('每股价值敏感性分析 (元)')
    
    for i in range(len(discount_rates)):
        for j in range(len(terminal_rates)):
            text = ax1.text(j, i, f'{values[i, j]:.1f}',
                          ha="center", va="center", color="black", fontsize=9)
    
    cbar1 = plt.colorbar(im1, ax=ax1)
    cbar1.set_label('每股价值 (元)')
    
    # 2. 安全边际热力图
    im2 = ax2.imshow(safety_margins, cmap='RdYlGn', aspect='auto',
                     vmin=-20, vmax=20)
    
    ax2.set_xticks(range(len(terminal_rates)))
    ax2.set_yticks(range(len(discount_rates)))
    ax2.set_xticklabels([f'{tr*100:.1f}%' for tr in terminal_rates])
    ax2.set_yticklabels([f'{dr*100:.1f}%' for dr in discount_rates])
    ax2.set_xlabel('终期增长率')
    ax2.set_ylabel('折现率')
    ax2.set_title('安全边际敏感性分析 (%)')
    
    for i in range(len(discount_rates)):
        for j in range(len(terminal_rates)):
            color = 'black' if -10 < safety_margins[i, j] < 10 else 'white'
            text = ax2.text(j, i, f'{safety_margins[i, j]:.1f}%',
                          ha="center", va="center", color=color, fontsize=9)
    
    cbar2 = plt.colorbar(im2, ax=ax2)
    cbar2.set_label('安全边际 (%)')
    
    plt.tight_layout()
    plt.show()

if 'sensitivity_df' in locals():
    plot_sensitivity_heatmap(sensitivity_df, current_price=28.0)

## 投资建议生成

In [ ]:
def generate_investment_recommendation(valuation_result: dict, current_price: float,
                                       sensitivity_df: pd.DataFrame) -> dict:
    """生成投资建议
    
    Args:
        valuation_result: DCF估值结果
        current_price: 当前股价
        sensitivity_df: 敏感性分析矩阵
    
    Returns:
        投资建议字典
    """
    intrinsic_value = valuation_result['每股价值(元)']
    safety_margin = (intrinsic_value - current_price) / current_price
    
    # 从敏感性矩阵中获取估值范围
    values = sensitivity_df.iloc[:, 1:].values
    min_value = values.min()
    max_value = values.max()
    
    # 投资评级
    if safety_margin > 0.3:
        rating = "强烈买入"
        risk_level = "低风险"
    elif safety_margin > 0.1:
        rating = "买入"
        risk_level = "中等风险"
    elif safety_margin > -0.1:
        rating = "持有"
        risk_level = "中等风险"
    else:
        rating = "卖出"
        risk_level = "高风险"
    
    return {
        '内在价值': f"{intrinsic_value:.2f} 元",
        '当前价格': f"{current_price:.2f} 元",
        '安全边际': f"{safety_margin*100:.1f}%",
        '估值范围': f"{min_value:.2f} - {max_value:.2f} 元",
        '投资评级': rating,
        '风险水平': risk_level,
        '买入点': f"{intrinsic_value * 0.8:.2f} 元" if safety_margin > 0 else "不建议买入",
        '目标价': f"{intrinsic_value * 1.2:.2f} 元" if safety_margin > 0 else "---",
        '止损点': f"{current_price * 0.85:.2f} 元" if safety_margin > 0 else "---"
    }

# 为所有公司生成投资建议
if 'sensitivity_df' in locals():
    print("投资建议摘要：")
    for result in valuation_results:
        recommendation = generate_investment_recommendation(result, 28.0, sensitivity_df)
        print(f"\n{result['公司名称']}：")
        for key, value in recommendation.items():
            print(f"  {key}: {value}")